In [260]:
import pandas as pd
import sqlite3
import os

print("Librairies chargées ✓")

Librairies chargées ✓


In [261]:
# Cellule 2 — Chargement du NOUVEAU dataset
df_raw = pd.read_csv(
    r"C:\Users\Jean Lavital\ALTER11\data\raw\players_data-2025_2026.csv",
    encoding="utf-8",
    sep=","
)

print(f"Shape : {df_raw.shape}")
print(f"Ligues : {df_raw['Comp'].value_counts().to_string()}")

Shape : (2779, 102)
Ligues : Comp
es La Liga            594
it Serie A            585
fr Ligue 1            554
eng Premier League    544
de Bundesliga         502


In [262]:
# # Cellule 3 — VERSION 2 avec stats défensives
rename_map = {
    "Player": "player_name",
    "Nation": "nation_raw",
    "Pos":    "position_raw",
    "Squad":  "team_name",
    "Comp":   "competition",
    "Age":    "age_raw",
    "Born":   "birth_year",
    "MP":     "matches_played",
    "Min":    "minutes",
    "90s":    "nineties",
    "Gls":    "goals",
    "Ast":    "assists",
    "Sh":     "shots",
    "SoT":    "shots_on_target",
    "Int":    "interceptions",
    "TklW":   "tackles_won",
    "CrdY":   "yellow_cards",
    "CrdR":   "red_cards",
    # Nouvelles colonnes
    "Fls":    "fouls_committed",
    "Fld":    "fouls_drawn",
    "Crs":    "crosses",
    "Mn/MP":  "min_per_match",
    "+/-":    "plus_minus",
    "PPM":    "points_per_match",
}

cols = {k: v for k, v in rename_map.items() if k in df_raw.columns}
df = df_raw[list(cols.keys())].rename(columns=cols).copy()

print(f"Colonnes gardées : {len(df.columns)}")
print(list(df.columns))

Colonnes gardées : 24
['player_name', 'nation_raw', 'position_raw', 'team_name', 'competition', 'age_raw', 'birth_year', 'matches_played', 'minutes', 'nineties', 'goals', 'assists', 'shots', 'shots_on_target', 'interceptions', 'tackles_won', 'yellow_cards', 'red_cards', 'fouls_committed', 'fouls_drawn', 'crosses', 'min_per_match', 'plus_minus', 'points_per_match']


In [263]:
# Cellule 4 — Nettoyage + filtre Ligue 1
# Supprimer les lignes parasites
df = df[df["player_name"] != "Player"].dropna(subset=["player_name"])
df = df[~df["team_name"].isin(["2TM", "3TM"])].copy()

# Nettoyer Nation : "fr FRA" → "FRA"
df["nation"] = df["nation_raw"].str.split().str[-1].str.upper()

# Nettoyer Position : "MF,FW" → "MF"
df["position"] = df["position_raw"].str.split(",").str[0].str.strip()
df["position"] = df["position"].map({"GK":"GK","DF":"DF","MF":"MF","FW":"FW"}).fillna("MF")

# Age en entier
df["age"] = pd.to_numeric(df["age_raw"], errors="coerce").fillna(0).astype(int)

# Toutes les stats en numérique
num_cols = ["birth_year","matches_played","minutes","nineties",
            "goals","assists","shots","shots_on_target",
            "interceptions","tackles_won","yellow_cards","red_cards"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

LIGUES = ['fr Ligue 1', 'es La Liga', 'it Serie A', 'eng Premier League', 'de Bundesliga']
df_l1 = df[df["competition"].isin(LIGUES)].copy()

print(f"Total joueurs 5 ligues : {len(df_l1)}")
print(f"Par ligue :\n{df_l1['competition'].value_counts()}")
print(f"U20 (≤20 ans) : {len(df_l1[df_l1['age'] <= 20])}")

Total joueurs 5 ligues : 2779
Par ligue :
competition
es La Liga            594
it Serie A            585
fr Ligue 1            554
eng Premier League    544
de Bundesliga         502
Name: count, dtype: int64
U20 (≤20 ans) : 373


In [264]:
# Cellule 5 — Construction des tables et export SQLite

# dim_team
dim_team = (
    df_l1[["team_name", "competition"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_team.insert(0, "team_id", range(1, len(dim_team) + 1))

# dim_player
df_l1 = df_l1.merge(dim_team[["team_id", "team_name"]], on="team_name", how="left")
dim_player = (
    df_l1[["player_name", "nation", "position", "birth_year", "age", "team_id"]]
    .drop_duplicates(subset=["player_name"])
    .reset_index(drop=True)
)
dim_player.insert(0, "player_id", range(1, len(dim_player) + 1))

# fact_stats
df_l1 = df_l1.merge(dim_player[["player_id", "player_name"]], on="player_name", how="left")
stat_cols = [
    "player_id", "matches_played", "minutes", "nineties",
    "goals", "assists", "shots", "shots_on_target",
    "interceptions", "tackles_won", "yellow_cards", "red_cards",
    "fouls_committed", "fouls_drawn", "min_per_match",
    "plus_minus", "points_per_match", "crosses"  # ← ajouté
]
fact_stats = df_l1[stat_cols].copy()
fact_stats.insert(0, "stat_id", range(1, len(fact_stats) + 1))

# Export SQLite
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")
dim_team.to_sql("dim_team", conn, if_exists="replace", index=False)
dim_player.to_sql("dim_player", conn, if_exists="replace", index=False)
fact_stats.to_sql("fact_stats", conn, if_exists="replace", index=False)
conn.close()

print(f"dim_team    : {len(dim_team)} clubs")
print(f"dim_player  : {len(dim_player)} joueurs")
print(f"fact_stats  : {len(fact_stats)} lignes")
print("✓ Base alter11.db créée !")

dim_team    : 96 clubs
dim_player  : 2627 joueurs
fact_stats  : 2779 lignes
✓ Base alter11.db créée !


In [265]:
# Cellule 6 — ALTERSCORE Attaquants U20 
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

query = """
    SELECT 
        p.player_name,
        p.age,
        t.team_name,
        f.minutes,
        f.matches_played,
        ROUND(f.minutes / NULLIF(f.matches_played, 0), 0)  AS min_par_match,
        f.goals,
        f.shots,
        ROUND(f.shots  / NULLIF(f.nineties, 0), 2)         AS tirs_p90,
        ROUND(f.shots_on_target / NULLIF(f.nineties, 0), 2) AS tirs_cadres_p90,
        ROUND(f.goals  / NULLIF(f.shots, 0) * 100, 1)      AS ratio_buts_tirs_pct,
        f.assists,

        -- ALTERSCORE : volume + audace + efficacité + régularité + bonus âge
        ROUND(
            (f.shots / NULLIF(f.nineties, 0)) * 2.5
          + (f.shots_on_target / NULLIF(f.nineties, 0)) * 2.0
          + (f.goals / NULLIF(f.shots, 0)) * 8
          + (f.minutes / NULLIF(f.matches_played, 0)) / 25
          + CASE WHEN p.age <= 18 THEN 4
                 WHEN p.age <= 19 THEN 3
                 WHEN p.age = 20  THEN 2
                 ELSE 1 END
        , 2) AS alterscore

    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team   t ON p.team_id   = t.team_id
    WHERE p.position = 'FW'
      AND p.age <= 20
      AND f.minutes >= 200
    ORDER BY alterscore DESC
    LIMIT 15
"""

top_fw = pd.read_sql(query, conn)
conn.close()

print("🔥 ALTERSCORE — Attaquants U20 Ligue 1 2025/2026\n")
print(top_fw.to_string(index=False))

🔥 ALTERSCORE — Attaquants U20 Ligue 1 2025/2026

          player_name  age           team_name  minutes  matches_played  min_par_match  goals  shots  tirs_p90  tirs_cadres_p90  ratio_buts_tirs_pct  assists  alterscore
Charalampos Kostoulas   18            Brighton      325              18           18.0      2     17      4.72             1.94                  0.0        1       19.69
              Endrick   19         Real Madrid     1055              14           75.0      5     40      3.42             1.97                  0.0        6       18.48
         Said El Mala   19                Köln     1787              32           55.0     12     71      3.57             1.41                  0.0        4       16.73
          Kader Meïté   18              Rennes      535              17           31.0      3     22      3.73             1.19                  0.0        2       16.69
    George Ilenikhena   19              Monaco      385              15           25.0      2     15 

In [266]:
print(list(df_raw.columns))

['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'G+A-PK', 'Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper', 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm', 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting', '90s_stats_shooting', 'Gls_stats_shooting', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'PK_stats_shooting', 'PKatt_stats_shooting', 'Rk_stats_playing_time', 'Nation_stats_playing_time', 'Pos_stats_playing_time', 'Comp_stats_playing_time', 'Age_stats_playing_time', 'Born_stats_playing_time', 'MP_stats_playing_time', 'Min_stats_playing_time', 'Mn/MP', 'Min%', '90s_s

In [267]:
for i, col in enumerate(df_raw.columns):
    print(f"{i:3} | {col}")

  0 | Rk
  1 | Player
  2 | Nation
  3 | Pos
  4 | Squad
  5 | Comp
  6 | Age
  7 | Born
  8 | MP
  9 | Starts
 10 | Min
 11 | 90s
 12 | Gls
 13 | Ast
 14 | G+A
 15 | G-PK
 16 | PK
 17 | PKatt
 18 | CrdY
 19 | CrdR
 20 | G+A-PK
 21 | Rk_stats_keeper
 22 | Nation_stats_keeper
 23 | Pos_stats_keeper
 24 | Comp_stats_keeper
 25 | Age_stats_keeper
 26 | Born_stats_keeper
 27 | MP_stats_keeper
 28 | Starts_stats_keeper
 29 | Min_stats_keeper
 30 | 90s_stats_keeper
 31 | GA
 32 | GA90
 33 | SoTA
 34 | Saves
 35 | Save%
 36 | W
 37 | D
 38 | L
 39 | CS
 40 | CS%
 41 | PKatt_stats_keeper
 42 | PKA
 43 | PKsv
 44 | PKm
 45 | Rk_stats_shooting
 46 | Nation_stats_shooting
 47 | Pos_stats_shooting
 48 | Comp_stats_shooting
 49 | Age_stats_shooting
 50 | Born_stats_shooting
 51 | 90s_stats_shooting
 52 | Gls_stats_shooting
 53 | Sh
 54 | SoT
 55 | SoT%
 56 | Sh/90
 57 | SoT/90
 58 | G/Sh
 59 | G/SoT
 60 | PK_stats_shooting
 61 | PKatt_stats_shooting
 62 | Rk_stats_playing_time
 63 | Nation_stats_pl

In [268]:
# Cellule 7 — ALTERSCORE Milieux U20
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

query_mf = """
    SELECT 
        p.player_name,
        p.age,
        t.team_name,
        f.minutes,
        f.matches_played,
        ROUND(f.minutes / NULLIF(f.matches_played, 0), 0)   AS min_par_match,
        f.goals,
        f.assists,
        f.shots,
        ROUND(f.shots    / NULLIF(f.nineties, 0), 2)        AS tirs_p90,
        ROUND(f.assists  / NULLIF(f.nineties, 0), 2)        AS passes_p90,
        f.interceptions,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2)   AS int_p90,

        -- ALTERSCORE MF : box-to-box, double contribution off+def + régularité + bonus âge
        ROUND(
            (f.shots    / NULLIF(f.nineties, 0)) * 1.5
          + (f.assists  / NULLIF(f.nineties, 0)) * 3.0
          + (f.goals    / NULLIF(f.nineties, 0)) * 4.0
          + (f.interceptions / NULLIF(f.nineties, 0)) * 2.0
          + (f.minutes  / NULLIF(f.matches_played, 0)) / 30
          + CASE WHEN p.age <= 18 THEN 4
                 WHEN p.age <= 19 THEN 3
                 WHEN p.age =  20 THEN 2
                 ELSE 1 END
        , 2) AS alterscore

    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team   t ON p.team_id   = t.team_id
    WHERE p.position = 'MF'
      AND p.age <= 20
      AND f.minutes >= 200
    ORDER BY alterscore DESC
    LIMIT 10
"""

top_mf = pd.read_sql(query_mf, conn)
conn.close()
print("🔥 ALTERSCORE — Milieux U20 Ligue 1 2025/2026\n")
print(top_mf.to_string(index=False))

🔥 ALTERSCORE — Milieux U20 Ligue 1 2025/2026

      player_name  age           team_name  minutes  matches_played  min_par_match  goals  assists  shots  tirs_p90  passes_p90  interceptions  int_p90  alterscore
     Lamine Yamal   18           Barcelona     2262              28           80.0     16       11    117      4.66        0.44              8     0.32       17.49
     Nathan Mbala   18                Metz      402              12           33.0      2        2     14      3.11        0.44              1     0.22       13.22
     Lennart Karl   18       Bayern Munich     1206              24           50.0      5        4     39      2.91        0.30              9     0.67       13.10
    Ethan Nwaneri   19             Arsenal      324               9           36.0      2        1      9      2.50        0.28              3     0.83       12.47
         Can Uzun   20 Eintracht Frankfurt     1097              20           54.0      8        4     40      3.28        0.33       

In [269]:
# Cellule 8 — ALTERSCORE Défenseurs U20
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

query_df = """
    SELECT 
        p.player_name,
        p.age,
        t.team_name,
        f.minutes,
        f.matches_played,
        ROUND(f.minutes / NULLIF(f.matches_played, 0), 0)     AS min_par_match,
        f.interceptions,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2)     AS int_p90,
        f.tackles_won,
        ROUND(f.tackles_won   / NULLIF(f.nineties, 0), 2)     AS tacles_p90,
        f.assists,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)           AS passes_p90,

        -- ALTERSCORE DF : solidité défensive + contribution offensive + régularité + bonus âge
        ROUND(
            (f.interceptions  / NULLIF(f.nineties, 0)) * 3.0
          + (f.tackles_won    / NULLIF(f.nineties, 0)) * 3.0
          + (f.assists        / NULLIF(f.nineties, 0)) * 2.5
          + (f.minutes / NULLIF(f.matches_played, 0)) / 25
          + CASE WHEN p.age <= 18 THEN 4
                 WHEN p.age <= 19 THEN 3
                 WHEN p.age =  20 THEN 2
                 ELSE 1 END
        , 2) AS alterscore

    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team   t ON p.team_id   = t.team_id
    WHERE p.position = 'DF'
      AND p.age <= 20
      AND f.minutes >= 200
    ORDER BY alterscore DESC
    LIMIT 10
"""

top_df = pd.read_sql(query_df, conn)
conn.close()
print("🔥 ALTERSCORE — Défenseurs U20 Europe 2025/2026\n")
print(top_df.to_string(index=False))

🔥 ALTERSCORE — Défenseurs U20 Europe 2025/2026

       player_name  age       team_name  minutes  matches_played  min_par_match  interceptions  int_p90  tackles_won  tacles_p90  assists  passes_p90  alterscore
Enzo Koffi Vinette   20        Le Havre      329              12           27.0              9     2.43           10        2.70        0         0.0       18.41
    Oliver Scarles   20 West Ham United      659              13           50.0             13     1.78           19        2.60        0         0.0       17.15
      Buba Sangaré   18           Elche      401               8           50.0              5     1.11           10        2.22        0         0.0       16.00
      Marius Louer   19          Angers      453               8           56.0              2     0.40           15        3.00        0         0.0       15.20
  Mahamadou Nagida   20          Rennes      640              17           37.0             19     2.68            9        1.27        0     

In [270]:
# Cellule 7 — ALTERSCORE GLOBAL (tous postes confondus)
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

query = """
    WITH base AS (
        SELECT
            p.player_name,
            p.age,
            p.position,
            t.team_name,
            f.minutes,
            f.matches_played,
            f.nineties,
            f.goals,
            f.assists,
            f.shots,
            f.shots_on_target,
            f.interceptions,
            f.tackles_won,

            -- composantes brutes
            ROUND(f.minutes / NULLIF(f.matches_played, 0), 1)          AS min_par_match,
            ROUND((f.goals + f.assists) / NULLIF(f.nineties, 0), 2)    AS impact_off_p90,
            ROUND((f.interceptions + f.tackles_won) / NULLIF(f.nineties, 0), 2) AS activite_def_p90,
            ROUND(f.shots / NULLIF(f.nineties, 0), 2)                  AS tirs_p90

        FROM fact_stats f
        JOIN dim_player p ON f.player_id = p.player_id
        JOIN dim_team   t ON p.team_id   = t.team_id
        WHERE p.age <= 20
          AND f.minutes >= 200
    )
    SELECT
        player_name,
        age,
        position,
        team_name,
        minutes,
        min_par_match,
        impact_off_p90,
        activite_def_p90,
        tirs_p90,

        -- ALTERSCORE GLOBAL sur 100
        ROUND(
            -- régularité /90 (max théorique ~90 min/match)
            (MIN(min_par_match, 90) / 90.0 * 100 * 0.30)
            -- impact offensif /90 (max théorique ~1.5)
          + (MIN(impact_off_p90, 1.5) / 1.5 * 100 * 0.30)
            -- activité défensive /90 (max théorique ~8)
          + (MIN(activite_def_p90, 8.0) / 8.0 * 100 * 0.20)
            -- bonus jeunesse (18→20 pts, 19→15, 20→10)
          + CASE WHEN age <= 18 THEN 20
                 WHEN age <= 19 THEN 15
                 WHEN age =  20 THEN 10
                 ELSE 5 END
        , 1) AS alterscore

    FROM base
    ORDER BY alterscore DESC
    LIMIT 20
"""

top = pd.read_sql(query, conn)
conn.close()

print("🔥 ALTERSCORE GLOBAL U20 — Europe 2025/2026\n")
print(top.to_string(index=False))

🔥 ALTERSCORE GLOBAL U20 — Europe 2025/2026

       player_name  age position           team_name  minutes  min_par_match  impact_off_p90  activite_def_p90  tirs_p90  alterscore
      Lamine Yamal   18       MF           Barcelona     2262           80.0            1.08              1.12      4.66        71.1
           Endrick   19       FW         Real Madrid     1055           75.0            0.94              0.85      3.42        60.9
      Yan Diomandé   19       FW          RB Leipzig     2303           74.0            0.74              1.29      1.95        57.7
       Robinio Vaz   19       FW           Marseille      379           27.0            1.43              1.67      1.90        56.8
      Lennart Karl   18       MF       Bayern Munich     1206           50.0            0.67              1.72      2.91        54.4
             Rayan   19       MF         Bournemouth      878           73.0            0.61              1.12      2.35        54.3
    Ayyoub Bouaddi   18  

In [271]:
# Ajouter la colonne min_pct à fact_stats
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")
try:
    conn.execute("ALTER TABLE fact_stats ADD COLUMN min_pct REAL DEFAULT 0")
    conn.commit()
    print("✓ Colonne min_pct ajoutée")
except:
    print("Colonne déjà existante")
conn.close()

✓ Colonne min_pct ajoutée


In [272]:
# Cellule — Ajout Min% dans fact_stats
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

# Récupérer Min% depuis le dataset brut filtré Ligue 1
df_minpct = df_raw[df_raw['Comp'] == 'fr Ligue 1'][['Player', 'Min%']].copy()
df_minpct.columns = ['player_name', 'min_pct']
df_minpct['min_pct'] = pd.to_numeric(df_minpct['min_pct'], errors='coerce').fillna(0)

# Récupérer les player_id
dim_player_db = pd.read_sql("SELECT player_id, player_name FROM dim_player", conn)
df_minpct = df_minpct.merge(dim_player_db, on='player_name', how='inner')

# Mettre à jour fact_stats
for _, row in df_minpct.iterrows():
    conn.execute(f"""
        UPDATE fact_stats 
        SET min_pct = {row['min_pct']} 
        WHERE player_id = {row['player_id']}
    """)

conn.commit()
conn.close()
print("✓ Min% ajouté dans fact_stats")

✓ Min% ajouté dans fact_stats


In [273]:
# Cellule — ALTERSCORE GLOBAL V2 (Min% au lieu de min_par_match)
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

query = """
    WITH base AS (
        SELECT
            p.player_name,
            p.age,
            p.position,
            t.team_name,
            f.minutes,
            f.min_pct,
            f.nineties,
            ROUND((f.goals + f.assists) / NULLIF(f.nineties, 0), 2)             AS impact_off_p90,
            ROUND((f.interceptions + f.tackles_won) / NULLIF(f.nineties, 0), 2) AS activite_def_p90,
            ROUND(f.shots / NULLIF(f.nineties, 0), 2)                           AS tirs_p90

        FROM fact_stats f
        JOIN dim_player p ON f.player_id = p.player_id
        JOIN dim_team   t ON p.team_id   = t.team_id
        WHERE p.age <= 20
          AND f.minutes >= 200
    )
    SELECT
        player_name,
        age,
        position,
        team_name,
        minutes,
        min_pct,
        impact_off_p90,
        activite_def_p90,
        tirs_p90,

        ROUND(
            -- régularité via Min% (titulaire indiscutable = 100%)
            (MIN(min_pct, 100) / 100.0 * 100 * 0.25)
            -- impact offensif /90
          + (MIN(impact_off_p90, 1.5) / 1.5 * 100 * 0.35)
            -- activité défensive /90
          + (MIN(activite_def_p90, 8.0) / 8.0 * 100 * 0.15)
            -- bonus jeunesse
          + CASE WHEN age <= 17 THEN 25
                 WHEN age <= 18 THEN 20
                 WHEN age <= 19 THEN 15
                 WHEN age =  20 THEN 10
                 ELSE 5 END
        , 1) AS alterscore

    FROM base
    ORDER BY alterscore DESC
    LIMIT 20
"""

top = pd.read_sql(query, conn)
conn.close()

print("🔥 ALTERSCORE GLOBAL V2 U20 — Ligue 1 2025/2026\n")
print(top.to_string(index=False))

🔥 ALTERSCORE GLOBAL V2 U20 — Ligue 1 2025/2026

          player_name  age position           team_name  minutes  min_pct  impact_off_p90  activite_def_p90  tirs_p90  alterscore
          Robinio Vaz   19       FW           Marseille      379     13.2            1.43              1.67      1.90        54.8
              Endrick   19       FW         Real Madrid     1055     36.6            0.94              0.85      3.42        47.7
         Lamine Yamal   18       MF           Barcelona     2262      0.0            1.08              1.12      4.66        47.3
         Nathan Mbala   18       MF                Metz      402     14.0            0.89              1.56      3.11        47.2
          Kader Meïté   18       FW              Rennes      535     18.6            0.85              0.51      3.73        45.4
       Ayyoub Bouaddi   18       MF               Lille     2186     75.9            0.04              2.55      0.62        44.7
         Senny Mayulu   19       MF Paris 

In [274]:
# ALTERSCORE V3 — avec coefficient de fiabilité
query = """
    WITH base AS (
        SELECT
            p.player_name,
            p.age,
            p.position,
            t.team_name,
            f.minutes,
            f.min_pct,
            f.nineties,
            ROUND((f.goals + f.assists) / NULLIF(f.nineties, 0), 2)             AS impact_off_p90,
            ROUND((f.interceptions + f.tackles_won) / NULLIF(f.nineties, 0), 2) AS activite_def_p90,
            ROUND(f.shots / NULLIF(f.nineties, 0), 2)                           AS tirs_p90
        FROM fact_stats f
        JOIN dim_player p ON f.player_id = p.player_id
        JOIN dim_team   t ON p.team_id   = t.team_id
        WHERE p.age <= 20
  AND f.minutes >= 200
  AND p.player_name != 'Robinio Vaz'
  AND p.position != 'GK'
    ),
    scored AS (
        SELECT *,
            ROUND(
             (MIN(min_pct, 100) / 100.0 * 100 * 0.20)
+ (MIN(impact_off_p90, 1.5) / 1.5 * 100 * 0.40)
+ (MIN(activite_def_p90, 8.0) / 8.0 * 100 * 0.10)
              + CASE WHEN age <= 17 THEN 25
                     WHEN age <= 18 THEN 20
                     WHEN age <= 19 THEN 15
                     WHEN age =  20 THEN 10
                     ELSE 5 END
            , 1) AS score_brut
        FROM base
    )
    SELECT
        player_name, age, position, team_name,
        minutes, min_pct, impact_off_p90, activite_def_p90,

        -- coefficient fiabilité : pénalise les petits échantillons
        -- 200 min → 0.60 | 500 min → 0.80 | 900 min → 0.95 | 1500+ min → 1.0
        ROUND(
            score_brut * MIN(1.0, 0.5 + (minutes / 3000.0))
        , 1) AS alterscore

    FROM scored
    ORDER BY alterscore DESC
    LIMIT 20
"""

conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")
top = pd.read_sql(query, conn)
conn.close()

print("🔥 ALTERSCORE V3 U20 — Ligue 1 2025/2026\n")
print(top.to_string(index=False))

🔥 ALTERSCORE V3 U20 — Ligue 1 2025/2026

       player_name  age position           team_name  minutes  min_pct  impact_off_p90  activite_def_p90  alterscore
      Lamine Yamal   18       MF           Barcelona     2262      0.0            1.08              1.12        50.2
           Endrick   19       FW         Real Madrid     1055     36.6            0.94              0.85        41.2
      Senny Mayulu   19       MF Paris Saint-Germain     1459     52.3            0.49              2.04        40.5
    Ayyoub Bouaddi   18       MF               Lille     2186     75.9            0.04              2.55        39.4
      Said El Mala   19       FW                Köln     1787      0.0            0.80              0.90        37.5
      Yan Diomandé   19       FW          RB Leipzig     2303      0.0            0.74              1.29        36.3
      Lennart Karl   18       MF       Bayern Munich     1206      0.0            0.67              1.72        36.1
Warren Zaïre-Emery   20

In [275]:
import subprocess
subprocess.run(["pip", "install", "requests", "Pillow"], check=True)
print("✓ Libs installées")

✓ Libs installées


In [276]:
API_KEY = "oHg5x9bvaJTzindcTNictsmS"

In [277]:
import requests
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
import os

API_KEY = "oHg5x9bvaJTzindcTNictsmS"  # colle ta nouvelle clé ici

def detourer_photo(url_photo, nom_joueur):
    """Télécharge d'abord la photo, puis l'envoie à remove.bg"""
    
    # Étape 1 : télécharger la photo localement
    headers = {"User-Agent": "Mozilla/5.0"}
    img_response = requests.get(url_photo, headers=headers)
    
    if img_response.status_code != 200:
        print(f"✗ Impossible de télécharger la photo de {nom_joueur}")
        return None
    
    # Étape 2 : envoyer le fichier à remove.bg
    response = requests.post(
        "https://api.remove.bg/v1.0/removebg",
        files={"image_file": ("photo.jpg", img_response.content, "image/jpeg")},
        data={"size": "auto"},
        headers={"X-Api-Key": API_KEY},
    )
    
    if response.status_code == 200:
        img_detoure = Image.open(BytesIO(response.content)).convert("RGBA")
        
        fond = Image.new("RGBA", (400, 500), (3, 8, 6, 255))
        draw = ImageDraw.Draw(fond)
        for x in range(0, 400, 20):
            draw.line([(x, 0), (x, 500)], fill=(0, 255, 106, 15), width=1)
        for y in range(0, 500, 20):
            draw.line([(0, y), (400, y)], fill=(0, 255, 106, 15), width=1)
        
        img_detoure.thumbnail((380, 440))
        x = (400 - img_detoure.width) // 2
        y = 20
        fond.paste(img_detoure, (x, y), img_detoure)
        
        draw.rectangle([(0, 460), (400, 462)], fill=(0, 255, 106, 200))
        draw.text((20, 468), nom_joueur.upper(), fill=(232, 245, 236, 255))
        
        os.makedirs("data/photos", exist_ok=True)
        chemin = f"data/photos/{nom_joueur.replace(' ', '_')}.png"
        fond.save(chemin)
        print(f"✓ {nom_joueur} — carte générée → {chemin}")
        return chemin
    else:
        print(f"✗ Erreur remove.bg {response.status_code} : {response.text}")
        return None

# Test sur Endrick
detourer_photo(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4a/Endrick_2024.jpg/220px-Endrick_2024.jpg",
    "Endrick"
)

✗ Impossible de télécharger la photo de Endrick


In [278]:
# Test — photo Ligue 1 déjà détourée !
url_endrick = "https://s3.eu-west-3.amazonaws.com/ligue1.image/players/2025/all/player_official_2025_143_577107-400x300.png"

img_response = requests.get(url_endrick, headers={"User-Agent": "Mozilla/5.0"})
print(f"Status : {img_response.status_code}")
print(f"Taille : {len(img_response.content)} bytes")

if img_response.status_code == 200:
    img = Image.open(BytesIO(img_response.content))
    print(f"Mode : {img.mode}")  # RGBA = fond transparent ✓
    print(f"Size : {img.size}")

Status : 200
Taille : 93325 bytes
Mode : P
Size : (800, 600)


In [279]:
# Génération carte ALTER11 avec photo Ligue 1 officielle
def generer_carte(nom_joueur, url_photo, score, pos, age, team):
    
    img_response = requests.get(url_photo, headers={"User-Agent": "Mozilla/5.0"})
    img = Image.open(BytesIO(img_response.content)).convert("RGBA")
    
    # Fond ALTER11
    fond = Image.new("RGBA", (400, 500), (2, 4, 12, 255))
    draw = ImageDraw.Draw(fond)
    
    # Grille verte
    # Grille bleue
    for x in range(0, 400, 20):
        draw.line([(x, 0), (x, 500)], fill=(0, 102, 255, 12), width=1)
    for y in range(0, 500, 20):
        draw.line([(0, y), (400, y)], fill=(0, 102, 255, 12), width=1)
    
    # Redimensionner et centrer le joueur
    img = img.resize((500, 375))
    x = (400 - img.width) // 2
    y = 60
    fond.paste(img, (x, y), img)

    # Fondu dégradé en bas du joueur
    fondu = Image.new("RGBA", (400, 200), (0, 0, 0, 0))
    draw_fondu = ImageDraw.Draw(fondu)
    for i in range(200):
        alpha = int((i / 200) ** 1.5 * 255)
        draw_fondu.rectangle([(0, i), (400, i+1)], fill=(3, 8, 6, alpha))
    fond.paste(fondu, (0, 240), fondu)
    
    # Bande inférieure
    draw.rectangle([(0, 440), (400, 500)], fill=(3, 5, 15, 240))
    draw.rectangle([(0, 440), (400, 442)], fill=(0, 102, 255, 200))
    
    # Textes
    draw.text((14, 448), nom_joueur.upper(), fill=(232, 245, 236, 255))
    draw.text((14, 468), f"{team} · {pos} · {age} ANS", fill=(74, 106, 82, 255))
    
    # Score badge
    draw.rectangle([(320, 448), (386, 490)], fill=(0, 102, 255, 30), outline=(0, 102, 255, 150))
    draw.text((328, 450), "ALTER", fill=(60, 80, 140, 255))
    draw.text((330, 463), str(score), fill=(0, 102, 255, 255))
    
    # Sauvegarder
    os.makedirs("../data/photos/alter11", exist_ok=True)
    chemin = f"../data/photos/alter11/{nom_joueur.replace(' ', '_')}.png"
    fond.save(chemin)
    print(f"✓ Carte générée → {chemin}")
    return chemin

# Test Endrick
generer_carte(
    nom_joueur="Endrick",
    url_photo="https://s3.eu-west-3.amazonaws.com/ligue1.image/players/2025/all/player_official_2025_143_577107-400x300.png",
    score=28.6,
    pos="FW",
    age=19,
    team="Lyon"
)

✓ Carte générée → ../data/photos/alter11/Endrick.png


'../data/photos/alter11/Endrick.png'

In [280]:
import re

def trouver_id_ligue1(nom_joueur):
    """Cherche l'ID Ligue 1 d'un joueur via une recherche sur le site"""
    
    nom_url = nom_joueur.lower().replace(' ', '-').replace('ï', 'i').replace('é', 'e').replace('è', 'e').replace('ë', 'e')
    url = f"https://ligue1.com/fr/player-sheet/{nom_url}"
    
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    
    # Chercher le pattern d'URL image dans le HTML
    matches = re.findall(r'player_official_2025_\d+_(\d+)-400x300\.png', response.text)
    
    if matches:
        player_id = matches[0]
        url_photo = f"https://s3.eu-west-3.amazonaws.com/ligue1.image/players/2025/all/player_official_2025_143_{player_id}-400x300.png"
        print(f"✓ {nom_joueur} → ID {player_id}")
        return url_photo
    else:
        print(f"✗ ID introuvable pour {nom_joueur}")
        return None

# Test sur les 6 joueurs
joueurs = ["endrick", "senny-mayulu", "ayyoub-bouaddi", "warren-zaire-emery", "prosper-peter", "desire-doue"]
for j in joueurs:
    trouver_id_ligue1(j)

✗ ID introuvable pour endrick
✗ ID introuvable pour senny-mayulu
✗ ID introuvable pour ayyoub-bouaddi
✗ ID introuvable pour warren-zaire-emery
✗ ID introuvable pour prosper-peter
✗ ID introuvable pour desire-doue


In [281]:
pip install selenium webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [282]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import re
import time

def get_photo_url_ligue1(nom_joueur):
    
    nom_url = (nom_joueur.lower()
        .replace(' ', '-').replace('é','e').replace('è','e')
        .replace('ë','e').replace('ï','i').replace('ô','o')
        .replace('î','i').replace('â','a').replace('à','a')
        .replace("'","-").replace('ü','u').replace('ç','c'))
    
    url = f"https://ligue1.com/fr/player-sheet/{nom_url}"
    
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    try:
        driver.get(url)
        
        # Attendre jusqu'à 10 secondes que les images chargent
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "img"))
            )
        except:
            pass
        
        time.sleep(5)  # attente supplémentaire pour le JS
        html = driver.page_source
        
        # Debug — affiche les 500 premiers caractères du body
        print(f"HTML length: {len(html)}")
        print(f"URL chargée: {driver.current_url}")
        
        # Cherche toutes les images
        imgs = driver.find_elements(By.TAG_NAME, "img")
        print(f"Images trouvées: {len(imgs)}")
        for img in imgs[:10]:
            src = img.get_attribute("src")
            if src:
                print(f"  → {src}")
        
        matches = re.findall(r'player_official_2025_\d+_(\d+)-400x300', html)
        if matches:
            player_id = matches[0]
            photo_url = f"https://s3.eu-west-3.amazonaws.com/ligue1.image/players/2025/all/player_official_2025_143_{player_id}-400x300.png"
            print(f"✓ {nom_joueur} → {photo_url}")
            return photo_url
        else:
            print("✗ Pattern non trouvé")
            return None
            
    finally:
        driver.quit()

get_photo_url_ligue1("Endrick")

HTML length: 201293
URL chargée: https://ligue1.com/fr/player-sheet/endrick/statistics
Images trouvées: 57
  → https://www.lfp.fr/assets/thumbnail_lfp_media_lo_sans_contour_horiz_rvb_0251570dee.png
  → https://ligue1.com/images/Logo_Ligue1.webp
  → https://ligue1.com/images/Logo_Ligue1.webp
  → https://ligue1.com/images/partnerships/MCDO.webp
  → https://ligue1.com/images/partnerships/BKT.webp
  → https://ligue1.com/images/Logo-Essilor.webp
  → https://ligue1.com/images/partnerships/DECATHLON.webp
  → https://ligue1.com/images/partnerships/LAPOSTE.webp
  → https://ligue1.com/images/partnerships/POINTP.webp
  → https://ligue1.com/images/partnerships/BEINSPORTS.webp
✗ Pattern non trouvé


In [283]:
# Dictionnaire manuel des IDs Ligue 1 — à compléter
# Format : "Nom Joueur" : "ID_ligue1"
PHOTOS_LIGUE1 = {
    "Endrick": "577107",
    # Les autres à trouver
}

def get_photo_s3(player_id, club_id="143"):
    """Construit l'URL S3 depuis l'ID joueur"""
    return f"https://s3.eu-west-3.amazonaws.com/ligue1.image/players/2025/all/player_official_2025_{club_id}_{player_id}-400x300.png"

In [284]:
print(df_raw['Comp'].value_counts())

Comp
es La Liga            594
it Serie A            585
fr Ligue 1            554
eng Premier League    544
de Bundesliga         502
Name: count, dtype: int64


In [285]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")
q = """
    SELECT p.player_name, p.age, t.team_name, 
           f.minutes, f.matches_played, f.goals, f.assists,
           ROUND(f.shots/NULLIF(f.nineties,0),2) as tirs_p90,
           f.min_pct
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    WHERE p.player_name LIKE '%Bamba%'
"""
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

    player_name  age team_name  minutes  matches_played  goals  assists  tirs_p90  min_pct
Abdoulaye Bamba   36    Angers      186               4      0        0      0.48      6.5
   Aladji Bamba   19    Monaco      689              16      0        1      0.13     23.9
  Mohamed Bamba   24   Lorient      682              18      1        2      1.71     23.7
    Bamba Dieng   26   Lorient     1115              20      9        1      3.23     38.7


In [286]:
# Toutes les stats d'Aladji Bamba dans le dataset brut
bamba = df_raw[df_raw['Player'] == 'Aladji Bamba']
print(bamba.T.to_string())  # affiche toutes les colonnes en vertical

                                    211
Rk                                  212
Player                     Aladji Bamba
Nation                           fr FRA
Pos                                  MF
Squad                            Monaco
Comp                         fr Ligue 1
Age                                19.0
Born                             2006.0
MP                                   16
Starts                                9
Min                                 689
90s                                 7.7
Gls                                   0
Ast                                   1
G+A                                   1
G-PK                                  0
PK                                    0
PKatt                                 0
CrdY                                  2
CrdR                                  0
G+A-PK                             0.13
Rk_stats_keeper                     NaN
Nation_stats_keeper                 NaN
Pos_stats_keeper                    NaN


In [287]:
# Quelles stats avancées sont disponibles ?
# On cherche : touches, passes, dribbles
cols_interessantes = [c for c in df_raw.columns if any(x in c.lower() for x in 
    ['touch', 'pass', 'carry', 'drib', 'prog', 'att', 'def', 'press'])]
print(f"Colonnes trouvées : {len(cols_interessantes)}")
for c in cols_interessantes:
    print(f"  {c}")

Colonnes trouvées : 3
  PKatt
  PKatt_stats_keeper
  PKatt_stats_shooting


In [288]:
# Vérification nouvelles colonnes sur Bamba
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

q = """
    SELECT p.player_name, p.age, t.team_name,
           f.minutes, f.tackles_won, f.interceptions,
           f.fouls_committed, f.fouls_drawn,
           f.plus_minus, f.points_per_match
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    WHERE p.player_name = 'Aladji Bamba'
"""
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

 player_name  age team_name  minutes  tackles_won  interceptions  fouls_committed  fouls_drawn  plus_minus  points_per_match
Aladji Bamba   19    Monaco      689           17              8               21           15         3.0              2.44


In [289]:
# Cellule — Recalcul min_pct pour toutes les ligues
# min_pct = minutes / (matches_played * 90) * 100
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

conn.execute("""
    UPDATE fact_stats 
    SET min_pct = ROUND(minutes * 100.0 / NULLIF(matches_played * 90.0, 0), 1)
    WHERE min_pct = 0 OR min_pct IS NULL
""")
conn.commit()
conn.close()
print("✓ min_pct recalculé pour tous les joueurs")

✓ min_pct recalculé pour tous les joueurs


In [290]:
# ALTERSCORE V4 — sur 10, sectorisé par poste
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

query = """
WITH base AS (
    SELECT
        p.player_name, p.age, p.position, t.team_name, t.competition,
        f.minutes, f.nineties, f.min_pct,
        f.goals, f.assists, f.shots, f.shots_on_target,
        f.tackles_won, f.interceptions,
        f.fouls_committed, f.fouls_drawn,
        f.plus_minus, f.points_per_match,
        f.matches_played,

        -- Stats /90
        ROUND(f.goals / NULLIF(f.nineties, 0), 2)          AS buts_p90,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)        AS passes_p90,
        ROUND(f.shots / NULLIF(f.nineties, 0), 2)          AS tirs_p90,
        ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2)    AS tacles_p90,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2)  AS int_p90,
        ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2)    AS fd_p90,

        -- Bonus jeunesse
        CASE 
            WHEN p.age <= 17 THEN 2.0
            WHEN p.age <= 18 THEN 1.7
            WHEN p.age <= 19 THEN 1.4
            WHEN p.age = 20  THEN 1.1
            ELSE 0.8
        END AS bonus_age,

        -- Coefficient fiabilité minutes
        MIN(1.0, 0.5 + (f.minutes / 3000.0)) AS coef_fiab

    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    WHERE p.age <= 20
      AND f.minutes >= 200
      AND p.position != 'GK'
      AND p.player_name != 'Robinio Vaz'
),
scored AS (
    SELECT *,
        CASE position
            -- ATTAQUANT : tirs, efficacité, création, régularité
            WHEN 'FW' THEN ROUND(
                (MIN(tirs_p90, 5.0) / 5.0 * 10 * 0.25)
              + (MIN(buts_p90, 1.0) / 1.0 * 10 * 0.25)
              + (MIN(passes_p90, 0.8) / 0.8 * 10 * 0.15)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.20 * 10 / 2.0)
            , 1)

            -- MILIEU : box-to-box, double contribution
            WHEN 'MF' THEN ROUND(
    CASE 
        -- Milieu défensif (peu de tirs)
        WHEN tirs_p90 < 0.5 THEN
            (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.30)
          + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.30)
          + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.15)
          + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
          + (bonus_age * 0.15 * 10 / 2.0)
        -- Milieu offensif
        ELSE
            (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.15)
          + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.15)
          + (MIN(buts_p90 + passes_p90, 1.0) / 1.0 * 10 * 0.25)
          + (MIN(tirs_p90, 3.0) / 3.0 * 10 * 0.20)
          + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
          + (bonus_age * 0.15 * 10 / 2.0)
    END
, 1)

            -- DÉFENSEUR : solidité, anticipation, impact
            WHEN 'DF' THEN ROUND(
                (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.25)
              + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.25)
              + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.10)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.25 * 10 / 2.0)
            , 1)
        END AS score_brut
    FROM base
)
SELECT
    player_name, age, position, team_name, competition,
    minutes, min_pct, tirs_p90, tacles_p90, int_p90,
    buts_p90, passes_p90,
    ROUND(score_brut * coef_fiab, 1) AS alterscore
FROM scored
WHERE score_brut IS NOT NULL
ORDER BY alterscore DESC
LIMIT 20
"""

top = pd.read_sql(query, conn)
conn.close()
print("🔵 ALTERSCORE V4 — U20 — 5 ligues\n")
print(top.to_string(index=False))

🔵 ALTERSCORE V4 — U20 — 5 ligues

       player_name  age position           team_name        competition  minutes  min_pct  tirs_p90  tacles_p90  int_p90  buts_p90  passes_p90  alterscore
      Lamine Yamal   18       MF           Barcelona         es La Liga     2262     89.8      4.66        0.80     0.32      0.64        0.44         7.1
      Said El Mala   19       FW                Köln      de Bundesliga     1787     62.0      3.57        0.60     0.30      0.60        0.20         6.0
      Lennart Karl   18       MF       Bayern Munich      de Bundesliga     1206     55.8      2.91        1.04     0.67      0.37        0.30         5.6
 Eli Junior Kroupi   19       MF         Bournemouth eng Premier League     1447     53.6      2.67        0.37     0.50      0.75        0.00         5.5
          Can Uzun   20       MF Eintracht Frankfurt      de Bundesliga     1097     60.9      3.28        0.74     0.25      0.66        0.33         5.5
      Yan Diomandé   19       FW    

In [291]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")
q = """
    SELECT p.player_name, p.age, p.position, t.team_name,
           f.minutes, f.tackles_won, f.interceptions,
           f.fouls_committed, f.fouls_drawn
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    WHERE p.player_name = 'Aladji Bamba'
"""
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

 player_name  age position team_name  minutes  tackles_won  interceptions  fouls_committed  fouls_drawn
Aladji Bamba   19       MF    Monaco      689           17              8               21           15


In [292]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")
q = """
-- Même requête V4 mais cherche Bamba spécifiquement
WITH base AS (
    SELECT p.player_name, p.age, p.position, t.team_name,
           f.minutes, f.nineties,
           ROUND(f.minutes * 100.0 / NULLIF(f.matches_played * 90.0, 0), 1) AS min_pct,
           ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2) AS tacles_p90,
           ROUND(f.interceptions / NULLIF(f.nineties, 0), 2) AS int_p90,
           ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2) AS fd_p90,
           ROUND(f.goals / NULLIF(f.nineties, 0), 2) AS buts_p90,
           ROUND(f.assists / NULLIF(f.nineties, 0), 2) AS passes_p90,
           ROUND(f.shots / NULLIF(f.nineties, 0), 2) AS tirs_p90,
           CASE WHEN p.age <= 19 THEN 1.4 ELSE 1.1 END AS bonus_age,
           MIN(1.0, 0.5 + (f.minutes / 3000.0)) AS coef_fiab
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    WHERE p.player_name = 'Aladji Bamba'
)
SELECT *,
    ROUND((
        (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.30)
      + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.30)
      + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.15)
      + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
      + (bonus_age * 0.15 * 10 / 2.0)
    ) * coef_fiab, 1) AS alterscore
FROM base
"""
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

 player_name  age position team_name  minutes  nineties  min_pct  tacles_p90  int_p90  fd_p90  buts_p90  passes_p90  tirs_p90  bonus_age  coef_fiab  alterscore
Aladji Bamba   19       MF    Monaco      689       7.7     47.8        2.21     1.04    1.95       0.0        0.13      0.13        1.4   0.729667         3.8


In [293]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

q = """
WITH base AS (
    SELECT
        p.player_name, p.age, p.position, t.team_name, t.competition,
        f.minutes, f.nineties,
        ROUND(f.minutes * 100.0 / NULLIF(f.matches_played * 90.0, 0), 1) AS min_pct,
        ROUND(f.goals / NULLIF(f.nineties, 0), 2)         AS buts_p90,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)       AS passes_p90,
        ROUND(f.shots / NULLIF(f.nineties, 0), 2)         AS tirs_p90,
        ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2)   AS tacles_p90,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2) AS int_p90,
        ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2)   AS fd_p90,
        CASE 
            WHEN p.age <= 17 THEN 2.0
            WHEN p.age <= 18 THEN 1.7
            WHEN p.age <= 19 THEN 1.4
            WHEN p.age = 20  THEN 1.1
            ELSE 0.8
        END AS bonus_age,
        MIN(1.0, 0.5 + (f.minutes / 3000.0)) AS coef_fiab
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    WHERE p.age <= 20
      AND f.minutes >= 200
      AND p.position != 'GK'
      AND p.player_name != 'Robinio Vaz'
),
scored AS (
    SELECT *,
        CASE position
            WHEN 'FW' THEN ROUND(
                (MIN(tirs_p90, 5.0) / 5.0 * 10 * 0.25)
              + (MIN(buts_p90, 1.0) / 1.0 * 10 * 0.25)
              + (MIN(passes_p90, 0.8) / 0.8 * 10 * 0.15)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.20 * 10 / 2.0)
            , 1)
            WHEN 'MF' THEN ROUND(
                CASE WHEN tirs_p90 < 0.5 THEN
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.30)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.30)
                  + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.15)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                ELSE
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.15)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.15)
                  + (MIN(buts_p90 + passes_p90, 1.0) / 1.0 * 10 * 0.25)
                  + (MIN(tirs_p90, 3.0) / 3.0 * 10 * 0.20)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                END
            , 1)
            WHEN 'DF' THEN ROUND(
                (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.25)
              + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.25)
              + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.10)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.25 * 10 / 2.0)
            , 1)
        END AS score_brut
    FROM base
),
final AS (
    SELECT *,
        ROUND(score_brut * coef_fiab, 1) AS alterscore
    FROM scored
    WHERE score_brut IS NOT NULL
)
SELECT
    ROW_NUMBER() OVER (ORDER BY alterscore DESC) AS rang,
    player_name, age, position, team_name, competition,
    minutes, alterscore
FROM final
ORDER BY alterscore DESC
LIMIT 30
"""

top = pd.read_sql(q, conn)
conn.close()
print("🔵 ALTERSCORE V4 — Top 30 U20 — 5 ligues\n")
print(top.to_string(index=False))

🔵 ALTERSCORE V4 — Top 30 U20 — 5 ligues

 rang        player_name  age position           team_name        competition  minutes  alterscore
    1       Lamine Yamal   18       MF           Barcelona         es La Liga     2262         7.1
    2       Said El Mala   19       FW                Köln      de Bundesliga     1787         6.0
    3       Lennart Karl   18       MF       Bayern Munich      de Bundesliga     1206         5.6
    4            Endrick   19       FW         Real Madrid         es La Liga     1055         5.5
    5  Eli Junior Kroupi   19       MF         Bournemouth eng Premier League     1447         5.5
    6           Can Uzun   20       MF Eintracht Frankfurt      de Bundesliga     1097         5.5
    7       Yan Diomandé   19       FW          RB Leipzig      de Bundesliga     2303         5.3
    8      Honest Ahanor   18       DF            Atalanta         it Serie A     1169         5.2
    9      Luka Vušković   19       DF        Hamburger SV      de B

In [294]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")
conn.execute("UPDATE dim_player SET position = 'FW' WHERE player_name = 'Lamine Yamal'")
conn.commit()
conn.close()
print("✓ Yamal corrigé en FW")

✓ Yamal corrigé en FW


In [295]:
MALUS_CLUBS = {

    # ══════════════════ LIGUE 1 ══════════════════
    'Paris Saint-Germain': 0.60,
    'Marseille':           0.60,
    'Lyon':                0.60,
    'Monaco':              0.60,
    'Lille':               0.80,
    'Lens':                0.80,
    'Rennes':              0.80,
    'Nice':                0.80,
    'Paris FC':            0.80,
    'Strasbourg':          0.80,
    'Auxerre':             1.00,
    'Brest':               1.00,
    'Nantes':              1.00,
    'Toulouse':            1.00,
    'Angers':              1.00,
    'Reims':               1.00,
    'Lorient':             1.00,
    'Metz':                1.00,

    # ══════════════════ PREMIER LEAGUE ══════════════════
    'Manchester City':     0.60,
    'Arsenal':             0.60,
    'Liverpool':           0.60,
    'Chelsea':             0.60,
    'Manchester Utd':      0.60,
    'Tottenham':           0.60,
    'Newcastle Utd':       0.80,
    'Aston Villa':         0.80,
    'Brighton':            0.80,
    'West Ham':            0.80,
    "Nott'ham Forest":     0.80,
    'Bournemouth':         1.00,
    'Brentford':           1.00,
    'Crystal Palace':      1.00,
    'Fulham':              1.00,
    'Wolves':              1.00,
    'Everton':             1.00,
    'Leeds United':        1.00,
    'Burnley':             1.00,
    'Sunderland':          1.00,

    # ══════════════════ LA LIGA ══════════════════
    'Real Madrid':         0.60,
    'Barcelona':           0.60,
    'Atletico Madrid':     0.60,
    'Sevilla':             0.80,
    'Real Betis':          0.80,
    'Athletic Club':       0.80,
    'Villarreal':          0.80,
    'Real Sociedad':       0.80,
    'Celta Vigo':          1.00,
    'Getafe':              1.00,
    'Girona':              1.00,
    'Las Palmas':          1.00,
    'Levante':             1.00,
    'Mallorca':            1.00,
    'Osasuna':             1.00,
    'Valladolid':          1.00,
    'Leganes':             1.00,
    'Espanyol':            1.00,
    'Alaves':              1.00,
    'Rayo Vallecano':      1.00,

    # ══════════════════ BUNDESLIGA ══════════════════
    'Bayern Munich':       0.60,
    'Borussia Dortmund':   0.60,
    'Leverkusen':          0.80,
    'RB Leipzig':          0.80,
    'Eintracht Frankfurt': 0.80,
    'Stuttgart':           0.80,
    'Hamburger SV':        0.80,
    'Augsburg':            1.00,
    'Freiburg':            1.00,
    'Heidenheim':          1.00,
    'Hoffenheim':          1.00,
    'Mainz 05':            1.00,
    'St. Pauli':           1.00,
    'Union Berlin':        1.00,
    'Wolfsburg':           1.00,
    'Werder Bremen':       1.00,
    'Köln':                1.00,
    "M'gladbach":          1.00,

    # ══════════════════ SERIE A ══════════════════
    'Inter':               0.60,
    'Juventus':            0.60,
    'AC Milan':            0.60,
    'Napoli':              0.60,
    'Roma':                0.80,
    'Lazio':               0.80,
    'Atalanta':            0.80,
    'Fiorentina':          0.80,
    'Bologna':             0.80,
    'Torino':              1.00,
    'Cagliari':            1.00,
    'Como':                1.00,
    'Empoli':              1.00,
    'Lecce':               1.00,
    'Monza':               1.00,
    'Parma':               1.00,
    'Udinese':             1.00,
    'Venezia':             1.00,
    'Verona':              1.00,
    'Genoa':               1.00,
}

print(f"✓ {len(MALUS_CLUBS)} clubs chargés")
print(f"  Malus fort  (0.60) : {sum(1 for v in MALUS_CLUBS.values() if v == 0.60)}")
print(f"  Malus moyen (0.80) : {sum(1 for v in MALUS_CLUBS.values() if v == 0.80)}")
print(f"  Pas de malus (1.00): {sum(1 for v in MALUS_CLUBS.values() if v == 1.00)}")

✓ 96 clubs chargés
  Malus fort  (0.60) : 19
  Malus moyen (0.80) : 26
  Pas de malus (1.00): 51


In [296]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

# Créer un DataFrame avec les malus
import pandas as pd
malus_df = pd.DataFrame([
    {'team_name': k, 'malus': v} for k, v in MALUS_CLUBS.items()
])
malus_df.to_sql('malus_clubs', conn, if_exists='replace', index=False)

q = """
WITH base AS (
    SELECT
        p.player_name, p.age, p.position, t.team_name, t.competition,
        f.minutes, f.nineties,
        ROUND(f.minutes * 100.0 / NULLIF(f.matches_played * 90.0, 0), 1) AS min_pct,
        ROUND(f.goals / NULLIF(f.nineties, 0), 2)         AS buts_p90,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)       AS passes_p90,
        ROUND(f.shots / NULLIF(f.nineties, 0), 2)         AS tirs_p90,
        ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2)   AS tacles_p90,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2) AS int_p90,
        ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2)   AS fd_p90,
        COALESCE(m.malus, 1.0)                             AS coef_club,
        CASE 
            WHEN p.age <= 17 THEN 2.0
            WHEN p.age <= 18 THEN 1.7
            WHEN p.age <= 19 THEN 1.4
            WHEN p.age = 20  THEN 1.1
            ELSE 0.8
        END AS bonus_age,
        MIN(1.0, 0.5 + (f.minutes / 3000.0)) AS coef_fiab
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    LEFT JOIN malus_clubs m ON t.team_name = m.team_name
    WHERE p.age <= 20
      AND f.minutes >= 200
      AND p.position != 'GK'
      AND p.player_name != 'Robinio Vaz'
),
scored AS (
    SELECT *,
        CASE position
            WHEN 'FW' THEN ROUND(
                (MIN(tirs_p90, 5.0) / 5.0 * 10 * 0.25)
              + (MIN(buts_p90, 1.0) / 1.0 * 10 * 0.25)
              + (MIN(passes_p90, 0.8) / 0.8 * 10 * 0.15)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.20 * 10 / 2.0)
            , 1)
            WHEN 'MF' THEN ROUND(
                CASE WHEN tirs_p90 < 0.5 THEN
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.30)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.30)
                  + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.15)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                ELSE
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.15)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.15)
                  + (MIN(buts_p90 + passes_p90, 1.0) / 1.0 * 10 * 0.25)
                  + (MIN(tirs_p90, 3.0) / 3.0 * 10 * 0.20)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                END
            , 1)
            WHEN 'DF' THEN ROUND(
                (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.25)
              + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.25)
              + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.10)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.25 * 10 / 2.0)
            , 1)
        END AS score_brut
    FROM base
)
SELECT
    ROW_NUMBER() OVER (ORDER BY score_brut * coef_fiab * coef_club DESC) AS rang,
    player_name, age, position, team_name, competition,
    minutes, coef_club,
    ROUND(score_brut * coef_fiab * coef_club, 1) AS alterscore
FROM scored
WHERE score_brut IS NOT NULL
ORDER BY alterscore DESC
LIMIT 30
"""

top = pd.read_sql(q, conn)
conn.close()
print("🔵 ALTERSCORE V5 — Top 30 U20 — 5 ligues — avec malus clubs\n")
print(top.to_string(index=False))

🔵 ALTERSCORE V5 — Top 30 U20 — 5 ligues — avec malus clubs

 rang        player_name  age position           team_name        competition  minutes  coef_club  alterscore
    1       Said El Mala   19       FW                Köln      de Bundesliga     1787        1.0         6.0
    2  Eli Junior Kroupi   19       MF         Bournemouth eng Premier League     1447        1.0         5.5
    3      Noahkai Banks   19       DF            Augsburg      de Bundesliga     1695        1.0         5.0
    4     Johan Manzambi   20       MF            Freiburg      de Bundesliga     1949        1.0         4.9
    5         Tylel Tati   18       DF              Nantes         fr Ligue 1     1721        1.0         4.9
    6        Carlos Espí   20       FW             Levante         es La Liga     1079        1.0         4.8
    7   Abdoul Coulibaly   18       DF       Werder Bremen      de Bundesliga     2030        1.0         4.8
    8    Jesus Rodríguez   20       MF                Como  

In [297]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

q = """
WITH base AS (
    SELECT
        p.player_name, p.age, p.position, t.team_name, t.competition,
        f.minutes, f.nineties,
        ROUND(f.minutes * 100.0 / NULLIF(f.matches_played * 90.0, 0), 1) AS min_pct,
        ROUND(f.goals / NULLIF(f.nineties, 0), 2)         AS buts_p90,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)       AS passes_p90,
        ROUND(f.shots / NULLIF(f.nineties, 0), 2)         AS tirs_p90,
        ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2)   AS tacles_p90,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2) AS int_p90,
        ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2)   AS fd_p90,
        COALESCE(m.malus, 1.0) AS coef_club,
        CASE 
            WHEN p.age <= 17 THEN 2.0
            WHEN p.age <= 18 THEN 1.7
            WHEN p.age <= 19 THEN 1.4
            WHEN p.age = 20  THEN 1.1
            ELSE 0.8
        END AS bonus_age,
        MIN(1.0, 0.5 + (f.minutes / 3000.0)) AS coef_fiab
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    LEFT JOIN malus_clubs m ON t.team_name = m.team_name
    WHERE p.age <= 20
      AND f.minutes >= 200
      AND p.position != 'GK'
      AND p.player_name != 'Robinio Vaz'
),
scored AS (
    SELECT *,
        CASE position
            WHEN 'FW' THEN ROUND(
                (MIN(tirs_p90, 5.0) / 5.0 * 10 * 0.25)
              + (MIN(buts_p90, 1.0) / 1.0 * 10 * 0.25)
              + (MIN(passes_p90, 0.8) / 0.8 * 10 * 0.15)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.20 * 10 / 2.0)
            , 1)
            WHEN 'MF' THEN ROUND(
                CASE WHEN tirs_p90 < 0.5 THEN
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.30)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.30)
                  + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.15)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                ELSE
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.15)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.15)
                  + (MIN(buts_p90 + passes_p90, 1.0) / 1.0 * 10 * 0.25)
                  + (MIN(tirs_p90, 3.0) / 3.0 * 10 * 0.20)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                END
            , 1)
            WHEN 'DF' THEN ROUND(
                (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.25)
              + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.25)
              + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.10)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.25 * 10 / 2.0)
            , 1)
        END AS score_brut
    FROM base
)
SELECT
    ROW_NUMBER() OVER (ORDER BY score_brut * coef_fiab * coef_club DESC) AS rang,
    player_name, age, position, team_name, competition,
    minutes, tirs_p90, tacles_p90, int_p90, buts_p90, passes_p90,
    ROUND(score_brut * coef_fiab * coef_club, 1) AS alterscore
FROM scored
WHERE score_brut IS NOT NULL
ORDER BY alterscore DESC
LIMIT 9
"""

top9 = pd.read_sql(q, conn)
conn.close()
print(top9.to_string(index=False))

 rang       player_name  age position     team_name        competition  minutes  tirs_p90  tacles_p90  int_p90  buts_p90  passes_p90  alterscore
    1      Said El Mala   19       FW          Köln      de Bundesliga     1787      3.57        0.60     0.30      0.60        0.20         6.0
    2 Eli Junior Kroupi   19       MF   Bournemouth eng Premier League     1447      2.67        0.37     0.50      0.75        0.00         5.5
    3     Noahkai Banks   19       DF      Augsburg      de Bundesliga     1695      0.48        1.33     0.96      0.05        0.05         5.0
    4    Johan Manzambi   20       MF      Freiburg      de Bundesliga     1949      2.26        0.55     1.24      0.23        0.14         4.9
    5        Tylel Tati   18       DF        Nantes         fr Ligue 1     1721      0.16        0.68     0.73      0.00        0.00         4.9
    6       Carlos Espí   20       FW       Levante         es La Liga     1079      3.67        0.42     0.42      0.75        0.

In [298]:
# Quelles colonnes du dataset brut sont exploitables pour les milieux ?
bamba = df_raw[df_raw['Player'] == 'Aladji Bamba']

# Affiche toutes les colonnes non-NaN de Bamba
cols_dispos = bamba.loc[:, bamba.notna().any()].columns.tolist()
print(f"Colonnes disponibles pour un MF ({len(cols_dispos)}) :")
for c in cols_dispos:
    val = bamba[c].values[0]
    print(f"  {c:35} → {val}")

Colonnes disponibles pour un MF (78) :
  Rk                                  → 212
  Player                              → Aladji Bamba
  Nation                              → fr FRA
  Pos                                 → MF
  Squad                               → Monaco
  Comp                                → fr Ligue 1
  Age                                 → 19.0
  Born                                → 2006.0
  MP                                  → 16
  Starts                              → 9
  Min                                 → 689
  90s                                 → 7.7
  Gls                                 → 0
  Ast                                 → 1
  G+A                                 → 1
  G-PK                                → 0
  PK                                  → 0
  PKatt                               → 0
  CrdY                                → 2
  CrdR                                → 0
  G+A-PK                              → 0.13
  Rk_stats_shooting           

In [299]:
print(df_raw[df_raw['Player'] == 'Aladji Bamba'][['Crs', 'Fls']].values)

[[ 4 21]]


In [300]:
print("Crs dans rename_map :", "Crs" in rename_map)
print("Fls dans rename_map :", "Fls" in rename_map)
print("Colonnes dans fact_stats :", pd.read_sql("PRAGMA table_info(fact_stats)", 
      sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db"))['name'].tolist())

Crs dans rename_map : True
Fls dans rename_map : True
Colonnes dans fact_stats : ['stat_id', 'player_id', 'matches_played', 'minutes', 'nineties', 'goals', 'assists', 'shots', 'shots_on_target', 'interceptions', 'tackles_won', 'yellow_cards', 'red_cards', 'fouls_committed', 'fouls_drawn', 'min_per_match', 'plus_minus', 'points_per_match', 'crosses', 'min_pct']


In [302]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

q = """
WITH base AS (
    SELECT
        p.player_name, p.age, p.position, t.team_name, t.competition,
        f.minutes, f.nineties,
        ROUND(f.minutes * 100.0 / NULLIF(f.matches_played * 90.0, 0), 1) AS min_pct,
        ROUND(f.goals / NULLIF(f.nineties, 0), 2)              AS buts_p90,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)            AS passes_p90,
        ROUND(f.shots / NULLIF(f.nineties, 0), 2)              AS tirs_p90,
        ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2)        AS tacles_p90,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2)      AS int_p90,
        ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2)        AS fd_p90,
        ROUND(f.fouls_committed / NULLIF(f.nineties, 0), 2)    AS fls_p90,
        ROUND(f.crosses / NULLIF(f.nineties, 0), 2)            AS crs_p90,
        ROUND(f.plus_minus / NULLIF(f.matches_played, 0), 2)   AS plus_minus_90,
        ROUND(f.points_per_match, 2)                           AS ppm,
        COALESCE(m.malus, 1.0)                                 AS coef_club,
        CASE 
            WHEN p.age <= 17 THEN 2.0
            WHEN p.age <= 18 THEN 1.7
            WHEN p.age <= 19 THEN 1.4
            WHEN p.age = 20  THEN 1.1
            ELSE 0.8
        END AS bonus_age,
        MIN(1.0, 0.5 + (f.minutes / 3000.0)) AS coef_fiab
    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    LEFT JOIN malus_clubs m ON t.team_name = m.team_name
    WHERE p.age <= 20
      AND f.minutes >= 200
      AND p.position != 'GK'
      AND p.player_name != 'Robinio Vaz'
),
scored AS (
    SELECT *,
        CASE position
            -- ATTAQUANT
            WHEN 'FW' THEN ROUND(
                (MIN(tirs_p90, 5.0) / 5.0 * 10 * 0.25)
              + (MIN(buts_p90, 1.0) / 1.0 * 10 * 0.25)
              + (MIN(passes_p90, 0.8) / 0.8 * 10 * 0.15)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
              + (bonus_age * 0.20 * 10 / 2.0)
            , 1)
            -- MILIEU
            WHEN 'MF' THEN ROUND(
                CASE WHEN tirs_p90 < 0.5 THEN
                    -- MF défensif
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.25)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.25)
                  + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.10)
                  + (MIN(fls_p90, 4.0) / 4.0 * 10 * 0.05)
                  + (MIN(ppm, 3.0) / 3.0 * 10 * 0.10)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                ELSE
                    -- MF offensif
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.10)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.10)
                  + (MIN(buts_p90 + passes_p90, 1.0) / 1.0 * 10 * 0.25)
                  + (MIN(tirs_p90, 3.0) / 3.0 * 10 * 0.20)
                  + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.10)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                END
            , 1)
            -- DÉFENSEUR
            WHEN 'DF' THEN ROUND(
                (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.22)
              + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.20)
              + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.08)
              + (MIN(fls_p90, 4.0) / 4.0 * 10 * 0.05)
              + (MIN(crs_p90, 3.0) / 3.0 * 10 * 0.10)
              + (MIN(ppm, 3.0) / 3.0 * 10 * 0.10)
              + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
              + (bonus_age * 0.15 * 10 / 2.0)
            , 1)
        END AS score_brut
    FROM base
)
SELECT
    ROW_NUMBER() OVER (ORDER BY score_brut * coef_fiab * coef_club DESC) AS rang,
    player_name, age, position, team_name, competition,
    minutes,
    ROUND(score_brut * coef_fiab * coef_club, 1) AS alterscore
FROM scored
WHERE score_brut IS NOT NULL
ORDER BY alterscore DESC
LIMIT 30
"""

top = pd.read_sql(q, conn)
conn.close()
print("🔵 ALTERSCORE V5 FINAL — Top 30 U20 — 5 ligues\n")
print(top.to_string(index=False))

🔵 ALTERSCORE V5 FINAL — Top 30 U20 — 5 ligues

 rang        player_name  age position           team_name        competition  minutes  alterscore
    1       Said El Mala   19       FW                Köln      de Bundesliga     1787         6.0
    2     Johan Manzambi   20       MF            Freiburg      de Bundesliga     1949         5.7
    3  Eli Junior Kroupi   19       MF         Bournemouth eng Premier League     1447         5.6
    4    Jesus Rodríguez   20       MF                Como         it Serie A     1585         5.1
    5    Bazoumana Touré   20       MF          Hoffenheim      de Bundesliga     2177         4.9
    6        Carlos Espí   20       FW             Levante         es La Liga     1079         4.8
    7        Mateus Mane   18       MF              Wolves eng Premier League     1549         4.8
    8           Can Uzun   20       MF Eintracht Frankfurt      de Bundesliga     1097         4.7
    9       Lamine Yamal   18       FW           Barcelona    

In [303]:
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")

q = """
WITH base AS (
    SELECT
        p.player_name, p.age, p.position, t.team_name, t.competition,
        f.minutes, f.nineties, f.matches_played,
        ROUND(f.minutes * 100.0 / NULLIF(f.matches_played * 90.0, 0), 1) AS min_pct,
        ROUND(f.goals / NULLIF(f.nineties, 0), 2)              AS buts_p90,
        ROUND(f.assists / NULLIF(f.nineties, 0), 2)            AS passes_p90,
        ROUND(f.shots / NULLIF(f.nineties, 0), 2)              AS tirs_p90,
        ROUND(f.tackles_won / NULLIF(f.nineties, 0), 2)        AS tacles_p90,
        ROUND(f.interceptions / NULLIF(f.nineties, 0), 2)      AS int_p90,
        ROUND(f.fouls_drawn / NULLIF(f.nineties, 0), 2)        AS fd_p90,
        ROUND(f.fouls_committed / NULLIF(f.nineties, 0), 2)    AS fls_p90,
        ROUND(f.crosses / NULLIF(f.nineties, 0), 2)            AS crs_p90,
        ROUND(f.points_per_match, 2)                           AS ppm,
        COALESCE(m.malus, 1.0)                                 AS coef_club,
        CASE 
            WHEN p.age <= 17 THEN 2.0
            WHEN p.age <= 18 THEN 1.7
            WHEN p.age <= 19 THEN 1.4
            WHEN p.age = 20  THEN 1.1
            ELSE 0.8
        END AS bonus_age,
        MIN(1.0, 0.5 + (f.minutes / 3000.0)) AS coef_fiab,

        -- Fix 2 : cut MF def/off plus intelligent
        CASE 
            WHEN (f.tackles_won + f.interceptions) / NULLIF(f.nineties, 0) 
               > (f.goals + f.assists) / NULLIF(f.nineties, 0) * 3 
            THEN 'MF_DEF' 
            ELSE 'MF_OFF' 
        END AS mf_type

    FROM fact_stats f
    JOIN dim_player p ON f.player_id = p.player_id
    JOIN dim_team t ON p.team_id = t.team_id
    LEFT JOIN malus_clubs m ON t.team_name = m.team_name
    WHERE p.age <= 20
      AND p.position != 'GK'
      AND p.player_name != 'Robinio Vaz'
),
scored AS (
    SELECT *,
        CASE position

            -- ATTAQUANT (min 300 min)
            WHEN 'FW' THEN
                CASE WHEN minutes < 300 THEN NULL ELSE
                ROUND(
                    (MIN(tirs_p90, 5.0) / 5.0 * 10 * 0.25)
                  + (MIN(buts_p90, 1.0) / 1.0 * 10 * 0.25)
                  + (MIN(passes_p90, 0.8) / 0.8 * 10 * 0.15)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.15)
                  + (MIN(ppm, 3.0) / 3.0 * 10 * 0.05)
                  + (bonus_age * 0.15 * 10 / 2.0)
                , 1) END

            -- MILIEU (min 400 min)
            WHEN 'MF' THEN
                CASE WHEN minutes < 400 THEN NULL ELSE
                ROUND(
                    CASE mf_type
                    WHEN 'MF_DEF' THEN
                        (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.25)
                      + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.25)
                      + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.10)
                      + (MIN(fls_p90, 4.0) / 4.0 * 10 * 0.05)
                      + (MIN(ppm, 3.0) / 3.0 * 10 * 0.10)
                      + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                      + (bonus_age * 0.15 * 10 / 2.0)
                    ELSE
                        (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.10)
                      + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.10)
                      + (MIN(buts_p90 + passes_p90, 1.0) / 1.0 * 10 * 0.25)
                      + (MIN(tirs_p90, 3.0) / 3.0 * 10 * 0.20)
                      + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.10)
                      + (MIN(ppm, 3.0) / 3.0 * 10 * 0.05)
                      + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                      + (bonus_age * 0.15 * 10 / 2.0)
                    END
                , 1) END

            -- DÉFENSEUR (min 500 min)
            WHEN 'DF' THEN
                CASE WHEN minutes < 500 THEN NULL ELSE
                ROUND(
                    (MIN(tacles_p90, 4.0) / 4.0 * 10 * 0.22)
                  + (MIN(int_p90, 3.0) / 3.0 * 10 * 0.20)
                  + (MIN(fd_p90, 3.0) / 3.0 * 10 * 0.08)
                  + (MIN(fls_p90, 4.0) / 4.0 * 10 * 0.05)
                  + (MIN(crs_p90, 3.0) / 3.0 * 10 * 0.10)
                  + (MIN(ppm, 3.0) / 3.0 * 10 * 0.10)
                  + (MIN(min_pct, 100) / 100.0 * 10 * 0.10)
                  + (bonus_age * 0.15 * 10 / 2.0)
                , 1) END

        END AS score_brut
    FROM base
),
final AS (
    SELECT *,
        ROUND(score_brut * coef_fiab * coef_club, 1) AS alterscore
    FROM scored
    WHERE score_brut IS NOT NULL
)

-- Fix 1 : Top par poste
SELECT rang, player_name, age, position, mf_type, team_name, competition, minutes, alterscore
FROM (
    SELECT 
        ROW_NUMBER() OVER (PARTITION BY position ORDER BY alterscore DESC) AS rang,
        *
    FROM final
)
WHERE rang <= 10
ORDER BY position, rang
"""

top = pd.read_sql(q, conn)
conn.close()
print("🔵 ALTERSCORE V6 — Top 10 par poste — U20 — 5 ligues\n")
for pos in ['FW', 'MF', 'DF']:
    print(f"\n{'═'*60}")
    print(f"  {pos}")
    print('═'*60)
    print(top[top['position']==pos].to_string(index=False))

🔵 ALTERSCORE V6 — Top 10 par poste — U20 — 5 ligues


════════════════════════════════════════════════════════════
  FW
════════════════════════════════════════════════════════════
 rang      player_name  age position mf_type   team_name   competition  minutes  alterscore
    1     Said El Mala   19       FW  MF_OFF        Köln de Bundesliga     1787         5.8
    2      Carlos Espí   20       FW  MF_OFF     Levante    es La Liga     1079         4.7
    3     Lamine Yamal   18       FW  MF_OFF   Barcelona    es La Liga     2262         4.7
    4     Yan Diomandé   19       FW  MF_OFF  RB Leipzig de Bundesliga     2303         4.2
    5 Christian Kofane   19       FW  MF_OFF  Leverkusen de Bundesliga     1162         3.7
    6    Sidiki Cherif   19       FW  MF_OFF      Angers    fr Ligue 1     1157         3.6
    7    Prosper Peter   18       FW  MF_OFF      Angers    fr Ligue 1     1228         3.6
    8          Endrick   19       FW  MF_OFF Real Madrid    es La Liga     1055    

In [304]:
# Vérif minutes Scarles
conn = sqlite3.connect(r"C:\Users\Jean Lavital\ALTER11\alter11.db")
q = "SELECT p.player_name, f.minutes FROM fact_stats f JOIN dim_player p ON f.player_id = p.player_id WHERE p.player_name LIKE '%Scarles%'"
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

   player_name  minutes
Oliver Scarles      659
